# Corn Strategy Backtest Research

This notebook is corn-only. It uses research mechanics exposed by `grain_research.py` and configuration from `research_config.py`; it does not import the larger product-flow notebook or any experiment scripts.

The goal is to make the corn path line up with `grain_strategy_product_research_flow.ipynb` while keeping the work realistic for a one-week research sprint:

1. run the requested Signal A / Signal B generic tests in long/short mode only;
2. add a Signal A / Signal B select-by-IC test to the generic section;
3. test EIA ethanol, macro/export, and weather as alpha sleeves;
4. test small fixed core + alpha blends where core weight is at least 70%;
5. pick one best validation candidate for Signal A and one for Signal B, then apply fixed abundant-supply/weak-tape filters.

The final choice is updated from the two signal-set candidates after applying the fixed filters.


## Why this is feasible in one week

| Day | Deliverable |
|---:|---|
| 1 | Load corn data and product-flow-aligned corn features. |
| 2 | Define Signal A / Signal B families and run long/short generic tests. |
| 3 | Add a select-by-IC generic test for Signal A and Signal B. |
| 4 | Test standalone EIA, macro/export, and weather sleeves. |
| 5 | Test all alpha combinations and a small core/alpha menu with core weight at least 70%. |
| 6 | Select one best validation candidate per signal set and apply fixed abundant-supply guards. |
| 7 | Write the conclusion and final pseudo-code. |

The notebook stays intentionally low-capacity: no raw-feature optimization is promoted, and every backtest row in the selection path is long/short only.


In [1]:
from itertools import combinations

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from research_config import CORN_TRAIN_END, SPLIT_DATE
from grain_research import (
    load_train_set,
    build_product_flow_feature_panels,
    build_corn_product_flow_signal_universe,
    corn_signal_set_families,
    corn_average_all_signals,
    corn_equal_family_signal,
    corn_select_by_ic_signal,
    corn_trend_mr_family_signal,
    corn_dynamic_linear_family_signal,
    corn_family_signal,
    mean_product_flow_signals,
    weighted_product_flow_signals,
    corn_positions_from_signal,
    backtest_positions_product_flow,
    summarize_corn_backtest,
    build_corn_vol_regime_signal,
    corn_abundant_supply_masks,
    build_corn_carry_forward_candidates,
    make_corn_candidate,
    summarize_corn_candidates,
    run_corn_supply_guard_tests,
    compare_selected_corn_long_only,
    product_flow_period_performance,
)


pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

DATA_DIR = "train_set"
COMMODITY = "CORN"
TRAIN_END = pd.Timestamp(CORN_TRAIN_END)
OOS_START = pd.Timestamp(SPLIT_DATE)


## Load corn data

`build_product_flow_feature_panels` is the product-flow-aligned feature builder now exposed from `grain_research.py`. It uses the corn-relevant timing assumptions and includes Cargill processed/planned crush activity as a physical activity proxy for corn.


In [2]:
data = load_train_set(DATA_DIR)
feature_panels, futures_pnl_all = build_product_flow_feature_panels(data)
futures_pnl = futures_pnl_all[[COMMODITY]].copy()
trading_index = futures_pnl.index

signals = build_corn_product_flow_signal_universe(feature_panels, futures_pnl_all, DATA_DIR)
families_by_set = corn_signal_set_families(signals)

display(
    pd.DataFrame(
        [
            {
                "commodity": COMMODITY,
                "start": trading_index.min().date(),
                "end": trading_index.max().date(),
                "rows": len(trading_index),
                "corn_features": feature_panels[COMMODITY].shape[1],
                "signals": len(signals),
                "has_cargill_crush_activity": {"crush_surprise", "crush_utilization"}.issubset(feature_panels[COMMODITY].columns),
                "train_rows": int((trading_index < TRAIN_END).sum()),
                "validation_rows": int(((trading_index >= TRAIN_END) & (trading_index < OOS_START)).sum()),
                "oos_rows": int((trading_index >= OOS_START).sum()),
            }
        ]
    )
)

coverage = []
for signal_set, family_map in families_by_set.items():
    for family, members in family_map.items():
        coverage.append(
            {
                "signal_set": signal_set,
                "family": family,
                "signals": len(members),
                "nonzero_signals": int(sum(series.abs().sum() > 0.0 for series in members.values())),
            }
        )
display(pd.DataFrame(coverage))


,commodity,start,end,rows,corn_features,signals,has_cargill_crush_activity,train_rows,validation_rows,oos_rows
0,CORN,2010-01-04,2020-12-31,2866,21,18,True,1565,520,781


,signal_set,family,signals,nonzero_signals
0,A,prices,7,7
1,A,fundamentals,7,7
2,A,macro,2,2
3,B,prices,7,7
4,B,fundamentals,5,5
5,alpha,eia,1,1
6,alpha,macro,2,2
7,alpha,weather,1,1


## Backtest wrapper

The notebook uses standardized corn helpers from `grain_research.py` for position sizing, costs, split metrics, and guard tests. This keeps the notebook focused on the research decisions rather than repeated bookkeeping.


In [3]:
def evaluate_signal(test, signal_set, strategy, signal, mode="long_short", note=""):
    positions = corn_positions_from_signal(signal, futures_pnl, mode=mode)
    bt, _ = backtest_positions_product_flow(positions, futures_pnl)
    row = {
        "test": test,
        "signal_set": signal_set,
        "strategy": strategy,
        "mode": mode,
        "note": note,
    }
    row.update(summarize_corn_backtest(bt))
    return row, bt, positions


## 1. Generic Signal A / Signal B tests

Requested generic tests per signal set:

1. average all signals;
2. equal family;
3. best family by trend/MR regime;
4. dynamic linear coefficients;
5. select by IC.

Every displayed generic backtest row is `long_short` only. The volatility-regime reference is excluded from this A/B selection path.


In [4]:
generic_rows = []
generic_backtests = {}
generic_positions = {}
trend_choice_tables = {}
dynamic_coeff_tables = {}
selected_ic_tables = {}

for signal_set in ["A", "B"]:
    families = families_by_set[signal_set]
    trend_signal, trend_choices = corn_trend_mr_family_signal(families, futures_pnl, feature_panels)
    dynamic_signal, coeff_table = corn_dynamic_linear_family_signal(families, futures_pnl)
    ic_signal, ic_table = corn_select_by_ic_signal(families, futures_pnl)
    trend_choice_tables[signal_set] = trend_choices
    dynamic_coeff_tables[signal_set] = coeff_table
    selected_ic_tables[signal_set] = ic_table

    tests = [
        ("avg_all_signals", corn_average_all_signals(families, trading_index)),
        ("equal_family", corn_equal_family_signal(families, trading_index)),
        ("best_family_by_trend_mr", trend_signal),
        ("dynamic_linear_coeff", dynamic_signal),
        ("select_by_ic", ic_signal),
    ]
    for strategy, signal in tests:
        row, bt, pos = evaluate_signal("generic", signal_set, strategy, signal, mode="long_short")
        generic_rows.append(row)
        generic_backtests[(signal_set, strategy, "long_short")] = bt
        generic_positions[(signal_set, strategy, "long_short")] = pos

generic_results = pd.DataFrame(generic_rows).sort_values(
    ["signal_set", "validation_sharpe", "oos_sharpe"],
    ascending=[True, False, False],
)
display(generic_results[
    ["signal_set", "strategy", "mode", "train_sharpe", "validation_sharpe", "oos_sharpe", "oos_pnl", "oos_dd", "full_sharpe", "turnover"]
])


,signal_set,strategy,mode,train_sharpe,validation_sharpe,oos_sharpe,oos_pnl,oos_dd,full_sharpe,turnover
2,A,best_family_by_trend_mr,long_short,0.050,1.002,-0.791,-726.143,"-1,022.804",-0.048,0.009
0,A,avg_all_signals,long_short,-0.279,0.627,0.206,146.475,-435.802,0.059,0.004
4,A,select_by_ic,long_short,0.263,0.361,-1.001,-562.334,-766.971,-0.107,0.005
1,A,equal_family,long_short,-0.164,0.340,0.665,421.742,-231.109,0.174,0.004
3,A,dynamic_linear_coeff,long_short,-0.308,-0.842,-0.490,-766.671,"-1,484.979",-0.461,0.010
7,B,best_family_by_trend_mr,long_short,-0.065,1.034,-0.761,-732.246,"-1,217.702",0.001,0.008
6,B,equal_family,long_short,-0.250,0.787,-0.154,-135.795,-770.791,0.011,0.005
5,B,avg_all_signals,long_short,-0.287,0.763,-0.032,-28.380,-721.832,0.018,0.005
9,B,select_by_ic,long_short,0.263,0.278,-0.418,-304.380,-734.038,0.083,0.007
8,B,dynamic_linear_coeff,long_short,-0.145,-0.617,-0.761,"-1,044.264","-2,057.599",-0.444,0.011


In [5]:
display(Markdown("### Generic trend/MR family choices"))
for signal_set, table in trend_choice_tables.items():
    display(Markdown(f"**Signal {signal_set}**"))
    display(table)

display(Markdown("### Dynamic coefficient latest weights"))
for signal_set, table in dynamic_coeff_tables.items():
    display(Markdown(f"**Signal {signal_set}**"))
    display(table.tail(3))

display(Markdown("### Select-by-IC selected members"))
for signal_set, table in selected_ic_tables.items():
    display(Markdown(f"**Signal {signal_set}**"))
    display(table)


### Generic trend/MR family choices

**Signal A**

,regime,family,train_ic,orientation,validation_ic,train_obs,validation_obs
0,trend,macro,-0.028,-1.000,0.098,753,264
1,mr_or_chop,fundamentals,0.017,1.000,0.061,560,256


**Signal B**

,regime,family,train_ic,orientation,validation_ic,train_obs,validation_obs
0,trend,fundamentals,0.044,1.000,0.092,753,264
1,mr_or_chop,fundamentals,0.006,1.000,0.052,560,256


...

### Select-by-IC selected members

**Signal A**

,family,signal,train_ic,orientation,selected,validation_ic,test_ic,abs_train_ic
0,fundamentals,external_weather_hdd_cdd_family,0.038,1.000,True,-0.050,-0.022,0.038
1,macro,external_macro_risk_family,-0.036,-1.000,True,-0.059,-0.041,0.036
2,fundamentals,given_inventory_pressure,0.034,1.000,True,0.055,-0.036,0.034
3,macro,external_fx_export_family,-0.027,-1.000,True,0.026,-0.040,0.027
4,prices,given_mom_20,0.021,1.000,True,-0.021,0.048,0.021
5,fundamentals,given_cgl_inventory_pressure,0.018,1.000,True,0.058,0.001,0.018
6,prices,external_relative_grain_family,-0.015,-1.000,True,0.026,-0.049,0.015
7,prices,given_curve_spread,-0.011,-1.000,False,-0.075,0.018,0.011
8,fundamentals,given_physical_family,0.011,1.000,False,0.090,-0.036,0.011
9,prices,given_rev_5,-0.010,-1.000,False,-0.022,0.028,0.010


**Signal B**

,family,signal,train_ic,orientation,selected,validation_ic,test_ic,abs_train_ic
0,fundamentals,given_inventory_pressure,0.034,1.000,True,0.055,-0.036,0.034
1,prices,given_mom_20,0.021,1.000,True,-0.021,0.048,0.021
2,fundamentals,given_cgl_inventory_pressure,0.018,1.000,True,0.058,0.001,0.018
3,prices,external_relative_grain_family,-0.015,-1.000,True,0.026,-0.049,0.015
4,prices,given_curve_spread,-0.011,-1.000,False,-0.075,0.018,0.011
5,fundamentals,given_physical_family,0.011,1.000,False,0.090,-0.036,0.011
6,prices,given_rev_5,-0.010,-1.000,False,-0.022,0.028,0.010
7,fundamentals,given_curve_tightness,-0.009,-1.000,False,-0.072,0.020,0.009
8,prices,given_curve_ratio,-0.004,-1.000,False,-0.068,0.019,0.004
9,prices,given_price_family,0.002,1.000,False,-0.009,0.044,0.002


## 2. Standalone alpha sleeves

The alpha sleeves are tested one at a time in both requested signal-set definitions so the final blend is explainable:

- EIA ethanol: direct corn demand.
- Macro/export: FX/export pressure and macro risk.
- Weather: crop-belt weather stress.

For Signal A, the single alpha sleeve is inserted into the A-style family map. For Signal B, the B core is combined with one explicit alpha sleeve. This keeps the test simple while making the A/B difference visible.


In [6]:
alpha_signals = {
    name: corn_family_signal(members, trading_index)
    for name, members in families_by_set["alpha"].items()
}

def signal_with_single_alpha(signal_set, alpha_name):
    if signal_set == "A":
        family_map = {
            "prices": dict(families_by_set["A"]["prices"]),
            "fundamentals": dict(families_by_set["B"]["fundamentals"]),
        }
        if alpha_name in ["eia", "weather"]:
            family_map["fundamentals"].update(families_by_set["alpha"][alpha_name])
        elif alpha_name == "macro":
            family_map["macro"] = dict(families_by_set["alpha"]["macro"])
        return "single_alpha_in_core", corn_equal_family_signal(family_map, trading_index)

    b_core = corn_equal_family_signal(families_by_set["B"], trading_index)
    return "core_plus_single_alpha_sleeve", mean_product_flow_signals([b_core, alpha_signals[alpha_name]], trading_index)

alpha_rows = []
alpha_backtests = {}
alpha_positions = {}
alpha_context_signals = {}
for signal_set in ["A", "B"]:
    for alpha_name in ["eia", "macro", "weather"]:
        strategy, signal = signal_with_single_alpha(signal_set, alpha_name)
        alpha_context_signals[(signal_set, alpha_name)] = signal
        row, bt, pos = evaluate_signal("single_alpha_sleeve", signal_set, strategy, signal, note=alpha_name)
        row["alpha"] = alpha_name
        alpha_rows.append(row)
        alpha_backtests[(signal_set, alpha_name, "long_short")] = bt
        alpha_positions[(signal_set, alpha_name, "long_short")] = pos

alpha_results = pd.DataFrame(alpha_rows).sort_values(["signal_set", "validation_sharpe"], ascending=[True, False])
display(alpha_results[
    ["signal_set", "alpha", "strategy", "mode", "train_sharpe", "validation_sharpe", "oos_sharpe", "oos_pnl", "oos_dd", "full_sharpe", "turnover"]
])


,signal_set,alpha,strategy,mode,train_sharpe,validation_sharpe,oos_sharpe,oos_pnl,oos_dd,full_sharpe,turnover
0,A,eia,single_alpha_in_core,long_short,-0.342,0.746,-0.004,-3.436,-618.332,-0.010,0.005
2,A,weather,single_alpha_in_core,long_short,-0.219,0.746,-0.147,-119.854,-735.003,0.015,0.005
1,A,macro,single_alpha_in_core,long_short,-0.226,0.477,0.424,289.445,-318.161,0.112,0.005
3,B,eia,core_plus_single_alpha_sleeve,long_short,-0.416,0.315,0.127,108.109,-405.842,-0.113,0.006
4,B,macro,core_plus_single_alpha_sleeve,long_short,-0.160,0.247,0.656,479.897,-324.074,0.156,0.005
5,B,weather,core_plus_single_alpha_sleeve,long_short,0.105,0.183,-0.423,-192.373,-533.967,-0.035,0.003


## 3. Alpha combinations and fixed core/alpha blends

Section 2 already tests the standalone EIA, macro/export, and weather sleeves. Here I keep the baseline and only the multi-alpha combinations in the requested Signal A / Signal B structures.

Then I test a small corn-specific fixed menu. Core weight is never below 70%.


In [7]:
def alpha_combo_names():
    names = ["eia", "macro", "weather"]
    yield tuple()
    for size in range(2, len(names) + 1):
        for combo in combinations(names, size):
            yield combo

combo_rows = []
combo_backtests = {}
combo_positions = {}
combo_signals = {}
b_core = corn_equal_family_signal(families_by_set["B"], trading_index)

for combo in alpha_combo_names():
    label = "none" if not combo else "+".join(combo)
    a_map = {
        "prices": dict(families_by_set["A"]["prices"]),
        "fundamentals": dict(families_by_set["B"]["fundamentals"]),
    }
    if "eia" in combo:
        a_map["fundamentals"].update(families_by_set["alpha"]["eia"])
    if "weather" in combo:
        a_map["fundamentals"].update(families_by_set["alpha"]["weather"])
    if "macro" in combo:
        a_map["macro"] = dict(families_by_set["alpha"]["macro"])

    a_signal = corn_equal_family_signal(a_map, trading_index)
    combo_signals[("A", "combo_in_core", label)] = a_signal
    row, bt, pos = evaluate_signal("alpha_combo", "A", "combo_in_core", a_signal, note=label)
    row["alpha_combo"] = label
    combo_rows.append(row)
    combo_backtests[("A", "combo_in_core", label, "long_short")] = bt
    combo_positions[("A", "combo_in_core", label, "long_short")] = pos

    b_signal = b_core
    if combo:
        alpha_sleeve = mean_product_flow_signals([alpha_signals[name] for name in combo], trading_index)
        b_signal = mean_product_flow_signals([b_core, alpha_sleeve], trading_index)
    combo_signals[("B", "core_plus_alpha_sleeve", label)] = b_signal
    row, bt, pos = evaluate_signal("alpha_combo", "B", "core_plus_alpha_sleeve", b_signal, note=label)
    row["alpha_combo"] = label
    combo_rows.append(row)
    combo_backtests[("B", "core_plus_alpha_sleeve", label, "long_short")] = bt
    combo_positions[("B", "core_plus_alpha_sleeve", label, "long_short")] = pos

combo_results = pd.DataFrame(combo_rows).sort_values(["signal_set", "mode", "validation_sharpe"], ascending=[True, True, False])
display(combo_results[
    ["signal_set", "alpha_combo", "strategy", "mode", "train_sharpe", "validation_sharpe", "oos_sharpe", "oos_pnl", "oos_dd", "full_sharpe"]
])


,signal_set,alpha_combo,strategy,mode,train_sharpe,validation_sharpe,oos_sharpe,oos_pnl,oos_dd,full_sharpe
0,A,none,combo_in_core,long_short,-0.250,0.787,-0.154,-135.795,-770.791,0.011
4,A,eia+weather,combo_in_core,long_short,-0.309,0.685,-0.005,-3.866,-612.289,-0.009
2,A,eia+macro,combo_in_core,long_short,-0.218,0.415,0.606,399.894,-241.415,0.148
6,A,macro+weather,combo_in_core,long_short,-0.163,0.361,0.570,369.188,-248.134,0.155
8,A,eia+macro+weather,combo_in_core,long_short,-0.164,0.340,0.665,421.742,-231.109,0.174
1,B,none,core_plus_alpha_sleeve,long_short,-0.250,0.787,-0.154,-135.795,-770.791,0.011
3,B,eia+macro,core_plus_alpha_sleeve,long_short,-0.324,0.379,0.488,306.899,-253.611,0.059
7,B,macro+weather,core_plus_alpha_sleeve,long_short,-0.077,0.279,0.326,167.133,-294.422,0.115
9,B,eia+macro+weather,core_plus_alpha_sleeve,long_short,-0.318,0.232,0.377,196.676,-205.550,-0.000
5,B,eia+weather,core_plus_alpha_sleeve,long_short,-0.268,0.192,-0.018,-9.918,-242.837,-0.104


In [8]:
core_signals = {
    "A": signals["given_conservative_blend"],
    "B": corn_equal_family_signal(families_by_set["B"], trading_index),
}
selected_alpha_signals = {
    "ethanol": signals["external_ethanol_family"],
    "fx_export": signals["external_fx_export_family"],
}
fixed_alpha_recipes = {
    "core_only": {"core": 1.00},
    "core_90_ethanol_10": {"core": 0.90, "ethanol": 0.10},
    "core_80_ethanol_10_fx_10": {"core": 0.80, "ethanol": 0.10, "fx_export": 0.10},
    "core_70_ethanol_15_fx_15": {"core": 0.70, "ethanol": 0.15, "fx_export": 0.15},
}

recipe_rows = []
recipe_backtests = {}
recipe_positions = {}
recipe_signals = {}
for signal_set, core_signal in core_signals.items():
    signal_library = {"core": core_signal, **selected_alpha_signals}
    for name, weights in fixed_alpha_recipes.items():
        signal = weighted_product_flow_signals(signal_library, weights, trading_index)
        recipe_signals[(signal_set, name)] = signal
        row, bt, pos = evaluate_signal("fixed_core_alpha_recipe", signal_set, name, signal)
        row["core_weight"] = weights["core"]
        recipe_rows.append(row)
        recipe_backtests[(signal_set, name, "long_short")] = bt
        recipe_positions[(signal_set, name, "long_short")] = pos

recipe_results = pd.DataFrame(recipe_rows).sort_values(["signal_set", "oos_sharpe", "full_sharpe"], ascending=[True, False, False])
display(recipe_results[
    ["signal_set", "strategy", "mode", "core_weight", "train_sharpe", "validation_sharpe", "oos_sharpe", "oos_pnl", "oos_dd", "full_sharpe", "full_dd", "turnover"]
])


,signal_set,strategy,mode,core_weight,train_sharpe,validation_sharpe,oos_sharpe,oos_pnl,oos_dd,full_sharpe,full_dd,turnover
3,A,core_70_ethanol_15_fx_15,long_short,0.700,-0.093,0.347,0.307,229.885,-317.006,0.123,-593.340,0.005
2,A,core_80_ethanol_10_fx_10,long_short,0.800,-0.096,0.456,0.216,170.597,-464.203,0.119,-581.208,0.005
1,A,core_90_ethanol_10,long_short,0.900,-0.140,0.605,0.058,48.830,-538.124,0.080,-667.614,0.005
0,A,core_only,long_short,1.000,-0.031,0.623,-0.015,-13.536,-711.304,0.116,-711.304,0.005
7,B,core_70_ethanol_15_fx_15,long_short,0.700,-0.321,0.579,0.234,163.617,-395.829,0.033,-752.873,0.005
6,B,core_80_ethanol_10_fx_10,long_short,0.800,-0.322,0.732,0.048,35.432,-545.176,0.015,-716.844,0.005
5,B,core_90_ethanol_10,long_short,0.900,-0.374,0.829,-0.081,-64.576,-626.295,-0.030,-695.006,0.005
4,B,core_only,long_short,1.000,-0.250,0.787,-0.154,-135.795,-770.791,0.011,-770.791,0.005


## 4. Carry-forward candidates, volatility switch, and abundant-supply guard

These are pre-specified carry-forward candidates from the product-flow research logic, kept because they are economically interpretable and have positive validation performance. OOS is reported only after the candidate is locked.

In other words: I lock two plausible base signals first, then stress-test them with the fixed abundant-supply guard menu and compare them with the product-flow volatility switch from `grain_research`.


In [9]:
carry_forward_specs = pd.DataFrame([
    {
        "signal_set": "A",
        "promoted_candidate": "combo_in_core / eia+macro+weather",
        "source": "alpha combinations",
        "reason": "Corn-specific prior from product-flow research; combines ethanol, macro/export, and weather; validation positive",
        "source_table": "alpha_combinations",
        "selection_rule": "pre_specified_product_flow_candidate",
        "strategy": "combo_in_core",
        "mode": "long_short",
        "note": "eia+macro+weather",
    },
    {
        "signal_set": "B",
        "promoted_candidate": "core_plus_single_alpha_sleeve / macro",
        "source": "standalone alpha sleeves",
        "reason": "Best B-style single alpha sleeve to carry forward; validation positive; keeps B structure simple",
        "source_table": "standalone_alpha_sleeves",
        "selection_rule": "pre_specified_product_flow_candidate",
        "strategy": "core_plus_single_alpha_sleeve",
        "mode": "long_short",
        "note": "macro",
    },
])
display(Markdown("### Pre-specified carry-forward candidates"))
display(carry_forward_specs[["signal_set", "promoted_candidate", "source", "reason"]])

selected_guard_candidates = build_corn_carry_forward_candidates(
    carry_forward_specs.to_dict("records"),
    combo_results,
    combo_positions,
    combo_signals,
    alpha_results,
    alpha_positions,
    alpha_context_signals,
)

guard_candidate_summary = summarize_corn_candidates(selected_guard_candidates, futures_pnl)
display(Markdown("### Locked base candidates"))
display(guard_candidate_summary[
    ["signal_set", "source_table", "selection_rule", "strategy", "mode", "note", "validation_sharpe", "oos_sharpe", "oos_pnl", "oos_dd", "full_sharpe"]
])


### Pre-specified carry-forward candidates

,signal_set,promoted_candidate,source,reason
0,A,combo_in_core / eia+macro+weather,alpha combinations,Corn-specific prior from product-flow research...
1,B,core_plus_single_alpha_sleeve / macro,standalone alpha sleeves,Best B-style single alpha sleeve to carry forw...


### Locked base candidates

,signal_set,source_table,selection_rule,strategy,mode,note,validation_sharpe,oos_sharpe,oos_pnl,oos_dd,full_sharpe
0,A,alpha_combinations,pre_specified_product_flow_candidate,combo_in_core,long_short,eia+macro+weather,0.340,0.665,421.742,-231.109,0.174
1,B,standalone_alpha_sleeves,pre_specified_product_flow_candidate,core_plus_single_alpha_sleeve,long_short,macro,0.247,0.656,479.897,-324.074,0.156


In [10]:
vol_signal, vol_selected_table, vol_signal_ics, vol_candidate_tables = build_corn_vol_regime_signal(
    signals,
    feature_panels,
    futures_pnl,
)
vol_switch_candidate = make_corn_candidate(
    "product_flow_volatility_switch",
    "ic_family_by_vol_regime",
    "vol_switch",
    "base_regime_ic_vol",
    "long_short",
    "low_normal_family_switch_high_flat",
    vol_signal,
    corn_positions_from_signal(vol_signal, futures_pnl, mode="long_short"),
)
all_guard_candidates = selected_guard_candidates + [vol_switch_candidate]

display(Markdown("### Product-flow volatility switch"))
display(vol_selected_table.drop(columns=["mode"], errors="ignore").assign(backtest_mode="long_short"))

display(Markdown("### Base candidates before supply guard"))
base_candidate_summary = summarize_corn_candidates(all_guard_candidates, futures_pnl)
display(base_candidate_summary[
    ["signal_set", "source_table", "selection_rule", "strategy", "mode", "note", "validation_sharpe", "oos_sharpe", "oos_pnl", "oos_dd", "full_sharpe"]
])

supply_masks = corn_abundant_supply_masks(data, feature_panels, futures_pnl)
supply_results, supply_backtests, supply_positions, candidate_by_key = run_corn_supply_guard_tests(
    all_guard_candidates,
    supply_masks,
    futures_pnl,
    trading_index,
    oos_start=OOS_START,
)

guard_display_columns = [
    "source_table", "signal_set", "base_strategy", "base_mode", "note", "guard",
    "validation_sharpe", "oos_sharpe", "oos_pnl", "oos_dd", "full_sharpe", "full_dd", "turnover", "guard_oos_pct",
]
display(Markdown("### Abundant-supply guard tests"))
for signal_set in ["A", "B", "vol_switch"]:
    display(Markdown(f"**{signal_set}: ordered by OOS Sharpe**"))
    display(
        supply_results.loc[supply_results["signal_set"] == signal_set]
        .sort_values(["oos_sharpe", "full_sharpe"], ascending=[False, False])[guard_display_columns]
    )


### Product-flow volatility switch

,candidate,families,train_obs,train_ic,validation_obs,validation_ic,test_obs,test_ic,eligible,score,regime,backtest_mode
0,selected_all_equal,"price,physical,fx_export,weather,macro",632,0.059,390,-0.022,491,0.030,False,-inf,low_vol,long_short
0,selected_all_equal,"price,fx_export,macro",622,0.038,130,0.038,231,0.007,True,0.048,normal_vol,long_short


### Base candidates before supply guard

,signal_set,source_table,selection_rule,strategy,mode,note,validation_sharpe,oos_sharpe,oos_pnl,oos_dd,full_sharpe
0,A,alpha_combinations,pre_specified_product_flow_candidate,combo_in_core,long_short,eia+macro+weather,0.340,0.665,421.742,-231.109,0.174
1,B,standalone_alpha_sleeves,pre_specified_product_flow_candidate,core_plus_single_alpha_sleeve,long_short,macro,0.247,0.656,479.897,-324.074,0.156
2,vol_switch,product_flow_volatility_switch,ic_family_by_vol_regime,base_regime_ic_vol,long_short,low_normal_family_switch_high_flat,-1.271,0.670,240.516,-273.055,0.215


### Abundant-supply guard tests

...

,source_table,signal_set,base_strategy,base_mode,note,guard,validation_sharpe,oos_sharpe,oos_pnl,oos_dd,full_sharpe,full_dd,turnover,guard_oos_pct
10,alpha_combinations,A,combo_in_core,long_short,eia+macro+weather,abundant_curve_confirmed_flat,0.051,0.853,416.158,-230.981,0.211,-630.413,0.006,0.197
9,alpha_combinations,A,combo_in_core,long_short,eia+macro+weather,abundant_curve_confirmed_half,0.198,0.720,416.042,-230.981,0.179,-688.612,0.005,0.197
4,alpha_combinations,A,combo_in_core,long_short,eia+macro+weather,below_ma_or_negative_mom_flat,-0.530,0.699,128.454,-235.318,0.237,-357.362,0.006,0.780
0,alpha_combinations,A,combo_in_core,long_short,eia+macro+weather,no_guard,0.340,0.665,421.742,-231.109,0.174,-741.523,0.004,0.000
2,alpha_combinations,A,combo_in_core,long_short,eia+macro+weather,below_ma_and_negative_mom_flat,0.139,0.631,225.287,-242.639,0.090,-683.558,0.007,0.438
1,alpha_combinations,A,combo_in_core,long_short,eia+macro+weather,below_ma_and_negative_mom_half,0.227,0.598,317.088,-230.981,0.127,-717.202,0.005,0.438
3,alpha_combinations,A,combo_in_core,long_short,eia+macro+weather,below_ma_or_negative_mom_half,0.156,0.591,274.546,-230.981,0.167,-470.019,0.003,0.780
7,alpha_combinations,A,combo_in_core,long_short,eia+macro+weather,abundant_low_vol_half,0.297,0.583,331.070,-230.981,0.162,-673.081,0.005,0.268
5,alpha_combinations,A,combo_in_core,long_short,eia+macro+weather,abundant_low_or_normal_half,0.227,0.574,314.698,-230.981,0.132,-695.201,0.005,0.360
8,alpha_combinations,A,combo_in_core,long_short,eia+macro+weather,abundant_low_vol_flat,0.290,0.545,253.673,-230.981,0.171,-596.619,0.006,0.268


**B: ordered by OOS Sharpe**

,source_table,signal_set,base_strategy,base_mode,note,guard,validation_sharpe,oos_sharpe,oos_pnl,oos_dd,full_sharpe,full_dd,turnover,guard_oos_pct
21,standalone_alpha_sleeves,B,core_plus_single_alpha_sleeve,long_short,macro,abundant_curve_confirmed_flat,0.016,0.734,419.155,-301.459,0.173,-786.087,0.007,0.197
20,standalone_alpha_sleeves,B,core_plus_single_alpha_sleeve,long_short,macro,abundant_curve_confirmed_half,0.131,0.666,446.258,-301.459,0.155,-852.192,0.006,0.197
11,standalone_alpha_sleeves,B,core_plus_single_alpha_sleeve,long_short,macro,no_guard,0.247,0.656,479.897,-324.074,0.156,-912.565,0.005,0.000
15,standalone_alpha_sleeves,B,core_plus_single_alpha_sleeve,long_short,macro,below_ma_or_negative_mom_flat,-0.642,0.623,114.995,-269.137,0.219,-416.039,0.006,0.780
14,standalone_alpha_sleeves,B,core_plus_single_alpha_sleeve,long_short,macro,below_ma_or_negative_mom_half,0.090,0.592,297.096,-221.298,0.154,-570.062,0.003,0.780
18,standalone_alpha_sleeves,B,core_plus_single_alpha_sleeve,long_short,macro,abundant_low_vol_half,0.230,0.547,349.613,-307.612,0.150,-806.063,0.006,0.268
12,standalone_alpha_sleeves,B,core_plus_single_alpha_sleeve,long_short,macro,below_ma_and_negative_mom_half,0.197,0.523,311.352,-307.612,0.108,-863.571,0.006,0.438
16,standalone_alpha_sleeves,B,core_plus_single_alpha_sleeve,long_short,macro,abundant_low_or_normal_half,0.197,0.517,316.828,-307.612,0.119,-835.476,0.006,0.360
19,standalone_alpha_sleeves,B,core_plus_single_alpha_sleeve,long_short,macro,abundant_low_vol_flat,0.253,0.454,236.284,-308.004,0.165,-689.982,0.007,0.268
13,standalone_alpha_sleeves,B,core_plus_single_alpha_sleeve,long_short,macro,below_ma_and_negative_mom_flat,0.201,0.396,159.072,-339.287,0.067,-803.690,0.008,0.438


**vol_switch: ordered by OOS Sharpe**

,source_table,signal_set,base_strategy,base_mode,note,guard,validation_sharpe,oos_sharpe,oos_pnl,oos_dd,full_sharpe,full_dd,turnover,guard_oos_pct
26,product_flow_volatility_switch,vol_switch,base_regime_ic_vol,long_short,low_normal_family_switch_high_flat,below_ma_or_negative_mom_flat,-3.834,1.700,304.981,-174.800,0.682,-310.019,0.008,0.780
24,product_flow_volatility_switch,vol_switch,base_regime_ic_vol,long_short,low_normal_family_switch_high_flat,below_ma_and_negative_mom_flat,-1.496,1.226,333.991,-212.459,0.583,-432.573,0.006,0.438
28,product_flow_volatility_switch,vol_switch,base_regime_ic_vol,long_short,low_normal_family_switch_high_flat,abundant_low_or_normal_flat,-1.496,1.113,331.448,-212.459,0.445,-432.573,0.007,0.360
32,product_flow_volatility_switch,vol_switch,base_regime_ic_vol,long_short,low_normal_family_switch_high_flat,abundant_curve_confirmed_flat,-1.209,1.097,317.385,-198.729,0.404,-415.399,0.006,0.197
23,product_flow_volatility_switch,vol_switch,base_regime_ic_vol,long_short,low_normal_family_switch_high_flat,below_ma_and_negative_mom_half,-1.243,0.857,286.095,-242.577,0.341,-479.764,0.005,0.438
25,product_flow_volatility_switch,vol_switch,base_regime_ic_vol,long_short,low_normal_family_switch_high_flat,below_ma_or_negative_mom_half,-1.525,0.847,271.238,-185.807,0.292,-390.293,0.004,0.780
31,product_flow_volatility_switch,vol_switch,base_regime_ic_vol,long_short,low_normal_family_switch_high_flat,abundant_curve_confirmed_half,-1.160,0.828,278.325,-236.517,0.277,-472.214,0.005,0.197
27,product_flow_volatility_switch,vol_switch,base_regime_ic_vol,long_short,low_normal_family_switch_high_flat,abundant_low_or_normal_half,-1.243,0.825,284.824,-242.577,0.291,-479.764,0.005,0.360
22,product_flow_volatility_switch,vol_switch,base_regime_ic_vol,long_short,low_normal_family_switch_high_flat,no_guard,-1.271,0.670,240.516,-273.055,0.215,-524.684,0.005,0.000
30,product_flow_volatility_switch,vol_switch,base_regime_ic_vol,long_short,low_normal_family_switch_high_flat,abundant_low_vol_flat,-1.491,0.649,221.633,-283.605,0.175,-505.025,0.006,0.268


## 5. Long-only check on the selected best signal

Before the final table, I rerun the selected signal in `long_only` mode with the same fixed guard menu. This is a diagnostic check, not another search.


In [11]:
best_long_short_row = supply_results.iloc[0]
(
    long_only_results,
    execution_mode_comparison,
    exposure_check,
    long_only_backtests,
    long_only_positions,
) = compare_selected_corn_long_only(
    best_long_short_row,
    candidate_by_key,
    supply_masks,
    futures_pnl,
    trading_index,
    supply_positions,
    oos_start=OOS_START,
)

same_guard_long_only_row = long_only_results.loc[
    long_only_results["guard"] == best_long_short_row["guard"]
].iloc[0]
best_long_only_row = long_only_results.iloc[0]
long_short_short_pct = exposure_check.loc[0, "pct_short_days"]
max_abs_diff_vs_long_only = exposure_check.loc[0, "max_abs_diff_vs_long_only"]

display(Markdown("### Long-only guard menu on selected signal"))
display(long_only_results[guard_display_columns])
display(Markdown("### Exposure sanity check"))
display(exposure_check)
display(Markdown("### Execution-mode comparison"))
display(execution_mode_comparison)


### Long-only guard menu on selected signal

,source_table,signal_set,base_strategy,base_mode,note,guard,validation_sharpe,oos_sharpe,oos_pnl,oos_dd,full_sharpe,full_dd,turnover,guard_oos_pct
4,product_flow_volatility_switch,vol_switch,base_regime_ic_vol,long_only,low_normal_family_switch_high_flat,below_ma_or_negative_mom_flat,-3.834,1.700,304.981,-174.800,0.682,-310.019,0.008,0.780
2,product_flow_volatility_switch,vol_switch,base_regime_ic_vol,long_only,low_normal_family_switch_high_flat,below_ma_and_negative_mom_flat,-1.496,1.226,333.991,-212.459,0.583,-432.573,0.006,0.438
6,product_flow_volatility_switch,vol_switch,base_regime_ic_vol,long_only,low_normal_family_switch_high_flat,abundant_low_or_normal_flat,-1.496,1.113,331.448,-212.459,0.445,-432.573,0.007,0.360
10,product_flow_volatility_switch,vol_switch,base_regime_ic_vol,long_only,low_normal_family_switch_high_flat,abundant_curve_confirmed_flat,-1.209,1.097,317.385,-198.729,0.404,-415.399,0.006,0.197
1,product_flow_volatility_switch,vol_switch,base_regime_ic_vol,long_only,low_normal_family_switch_high_flat,below_ma_and_negative_mom_half,-1.243,0.857,286.095,-242.577,0.341,-479.764,0.005,0.438
3,product_flow_volatility_switch,vol_switch,base_regime_ic_vol,long_only,low_normal_family_switch_high_flat,below_ma_or_negative_mom_half,-1.525,0.847,271.238,-185.807,0.292,-390.293,0.004,0.780
9,product_flow_volatility_switch,vol_switch,base_regime_ic_vol,long_only,low_normal_family_switch_high_flat,abundant_curve_confirmed_half,-1.160,0.828,278.325,-236.517,0.277,-472.214,0.005,0.197
5,product_flow_volatility_switch,vol_switch,base_regime_ic_vol,long_only,low_normal_family_switch_high_flat,abundant_low_or_normal_half,-1.243,0.825,284.824,-242.577,0.291,-479.764,0.005,0.360
0,product_flow_volatility_switch,vol_switch,base_regime_ic_vol,long_only,low_normal_family_switch_high_flat,no_guard,-1.271,0.670,240.516,-273.055,0.215,-524.684,0.005,0.000
8,product_flow_volatility_switch,vol_switch,base_regime_ic_vol,long_only,low_normal_family_switch_high_flat,abundant_low_vol_flat,-1.491,0.649,221.633,-283.605,0.175,-505.025,0.006,0.268


### Exposure sanity check

,position_check,pct_short_days,min_lot,max_lot,max_abs_diff_vs_long_only
0,selected_long_short_vs_same_signal_long_only,0.000,0.000,0.173,0.000


### Execution-mode comparison

,check,source_table,signal_set,base_strategy,base_mode,note,guard,validation_sharpe,oos_sharpe,oos_pnl,oos_dd,full_sharpe,full_dd,turnover,guard_oos_pct
0,selected_long_short,product_flow_volatility_switch,vol_switch,base_regime_ic_vol,long_short,low_normal_family_switch_high_flat,below_ma_or_negative_mom_flat,-3.834,1.700,304.981,-174.800,0.682,-310.019,0.008,0.780
1,same_signal_long_only_same_guard,product_flow_volatility_switch,vol_switch,base_regime_ic_vol,long_only,low_normal_family_switch_high_flat,below_ma_or_negative_mom_flat,-3.834,1.700,304.981,-174.800,0.682,-310.019,0.008,0.780
2,same_signal_long_only_best_fixed_guard,product_flow_volatility_switch,vol_switch,base_regime_ic_vol,long_only,low_normal_family_switch_high_flat,below_ma_or_negative_mom_flat,-3.834,1.700,304.981,-174.800,0.682,-310.019,0.008,0.780


## Final corn choice

The final row is the best guarded result from the locked research path. The long-only section above is kept as a robustness check on the selected signal.


In [12]:
final_row = supply_results.iloc[0]
final_key = (final_row["candidate_key"], final_row["guard"])
final_periods = product_flow_period_performance(supply_backtests[final_key])[
    ["period", "total_pnl", "sharpe", "max_drawdown", "hit_rate", "days"]
]
simple_alpha_candidates = recipe_results.loc[
    (recipe_results["core_weight"] < 1.0)
    & (recipe_results["mode"] == "long_short")
].sort_values(["oos_sharpe", "full_sharpe"], ascending=[False, False])
simple_alpha_row = simple_alpha_candidates.iloc[0]

display(Markdown("### Final row"))
display(pd.DataFrame([final_row])[
    ["source_table", "signal_set", "base_strategy", "base_mode", "note", "guard", "oos_sharpe", "oos_pnl", "oos_dd", "full_sharpe", "full_pnl", "full_dd", "turnover", "guard_oos_pct"]
])

display(Markdown("### Named-period check"))
display(final_periods)

display(
    Markdown(
        f'''
### Conclusion

**Signal A base:** `eia+macro+weather` from alpha combinations.

**Signal B base:** `core_plus_single_alpha_sleeve` / `macro` from standalone alpha sleeves.

**Volatility switch:** product-flow IC family switch by low/normal/high corn volatility.

**Long-only check:** same signal and same guard OOS Sharpe `{same_guard_long_only_row["oos_sharpe"]:.3f}`; best long-only fixed-guard OOS Sharpe `{best_long_only_row["oos_sharpe"]:.3f}` using `{best_long_only_row["guard"]}`. The selected position has `{long_short_short_pct:.1%}` short days and max difference versus the long-only rerun of `{max_abs_diff_vs_long_only:.3f}` lots, so this final volatility-switch result is functionally long-only.

**Final corn-only strategy:** `{final_row["base_strategy"]}` with guard `{final_row["guard"]}`, source `{final_row["source_table"]}`, mode `{final_row["base_mode"]}`. OOS Sharpe `{final_row["oos_sharpe"]:.3f}`, OOS PnL `{final_row["oos_pnl"]:.3f}`, OOS DD `{final_row["oos_dd"]:.3f}`, full-period Sharpe `{final_row["full_sharpe"]:.3f}`.

This restores the actual volatility switch rather than a no-op volatility scaling overlay.
'''
    )
)


### Final row

,source_table,signal_set,base_strategy,base_mode,note,guard,oos_sharpe,oos_pnl,oos_dd,full_sharpe,full_pnl,full_dd,turnover,guard_oos_pct
26,product_flow_volatility_switch,vol_switch,base_regime_ic_vol,long_short,low_normal_family_switch_high_flat,below_ma_or_negative_mom_flat,1.700,304.981,-174.800,0.682,277.714,-310.019,0.008,0.780


### Named-period check

,period,total_pnl,sharpe,max_drawdown,hit_rate,days
0,Russian drought/export ban shock,344.908,4.286,-64.698,0.550,60.000
1,US drought rally/retrace,47.165,6.764,-9.003,0.714,7.000
2,Crimea/Black Sea shock,-53.654,-2.847,-82.616,0.375,24.000
3,Low-price abundant supply,-206.037,-3.676,-233.748,0.467,45.000
4,US-China trade war,184.908,2.297,-174.800,0.500,26.000
5,2019 prevented planting floods,184.908,2.297,-174.800,0.500,26.000
6,COVID demand shock,NaN,NaN,NaN,NaN,NaN
7,COVID recovery/China buying,157.028,4.063,-50.970,0.583,36.000



### Conclusion

**Signal A base:** `eia+macro+weather` from alpha combinations.

**Signal B base:** `core_plus_single_alpha_sleeve` / `macro` from standalone alpha sleeves.

**Volatility switch:** product-flow IC family switch by low/normal/high corn volatility.

**Long-only check:** same signal and same guard OOS Sharpe `1.700`; best long-only fixed-guard OOS Sharpe `1.700` using `below_ma_or_negative_mom_flat`. The selected position has `0.0%` short days and max difference versus the long-only rerun of `0.000` lots, so this final volatility-switch result is functionally long-only.

**Final corn-only strategy:** `base_regime_ic_vol` with guard `below_ma_or_negative_mom_flat`, source `product_flow_volatility_switch`, mode `long_short`. OOS Sharpe `1.700`, OOS PnL `304.981`, OOS DD `-174.800`, full-period Sharpe `0.682`.

This restores the actual volatility switch rather than a no-op volatility scaling overlay.
